In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

matches = pd.read_csv('../data/dataraw/matches.csv')
deliveries = pd.read_csv('../data/dataraw/deliveries.csv')

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)
print("\nMatches columns:", matches.columns.tolist())
print("\nDeliveries columns:", deliveries.columns.tolist())

Matches shape: (1095, 20)
Deliveries shape: (260920, 17)

Matches columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

Deliveries columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']


In [2]:
# Check missing values
print("=== MATCHES MISSING VALUES ===")
print(matches.isnull().sum())
print("\n=== DELIVERIES MISSING VALUES ===")
print(deliveries.isnull().sum())

=== MATCHES MISSING VALUES ===
id                    0
season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

=== DELIVERIES MISSING VALUES ===
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder  

In [3]:
# Standardize team names
team_name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}

for col in ['team1', 'team2', 'winner', 'toss_winner']:
    if col in matches.columns:
        matches[col] = matches[col].replace(team_name_map)

for col in ['batting_team', 'bowling_team']:
    if col in deliveries.columns:
        deliveries[col] = deliveries[col].replace(team_name_map)

print("Team names standardized.")

Team names standardized.


In [4]:
# Drop abandoned matches
matches = matches[matches['result'] != 'no result']

# Fill missing method column
if 'method' in matches.columns:
    matches['method'] = matches['method'].fillna('Normal')

# Convert date
matches['date'] = pd.to_datetime(matches['date'])
matches['season'] = matches['date'].dt.year

print("Matches after cleaning:", matches.shape)
print(matches['season'].value_counts().sort_index())

Matches after cleaning: (1090, 20)
season
2008    58
2009    57
2010    60
2011    72
2012    74
2013    76
2014    60
2015    57
2016    60
2017    59
2018    60
2019    59
2020    60
2021    60
2022    74
2023    73
2024    71
Name: count, dtype: int64


In [5]:
# Feature engineering on deliveries
deliveries['is_boundary'] = deliveries['batsman_runs'].apply(
    lambda x: 1 if x in [4, 6] else 0
)
deliveries['is_wicket'] = deliveries['player_dismissed'].notna().astype(int)

def get_phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 14:
        return 'Middle'
    else:
        return 'Death'

deliveries['phase'] = deliveries['over'].apply(get_phase)

print("New columns added:")
print(deliveries[['batsman_runs','is_boundary','is_wicket','phase']].head(10))

New columns added:
   batsman_runs  is_boundary  is_wicket      phase
0             0            0          0  Powerplay
1             0            0          0  Powerplay
2             0            0          0  Powerplay
3             0            0          0  Powerplay
4             0            0          0  Powerplay
5             0            0          0  Powerplay
6             0            0          0  Powerplay
7             0            0          0  Powerplay
8             4            1          0  Powerplay
9             4            1          0  Powerplay


In [6]:
# Save cleaned files
matches.to_csv('../data/datacleaned/matches_cleaned.csv', index=False)
deliveries.to_csv('../data/datacleaned/deliveries_final.csv', index=False)
print("Files saved to datacleaned folder!")

Files saved to datacleaned folder!
